# **Filtrado de Complaints por PROD_TYPE**

Este notebook filtra los complaints para:
1. Mantener solo PROD_TYPE = 'V' (vehicles) y 'T' (tires)
2. Identificar marcas exclusivas de 'C' (Child Seats) y 'E' (Equipment)
3. Exportar lista de marcas excluibles para Recalls e Investigations


In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os

# Configuración de rutas (ajustadas para notebook)
BASE_DIR = Path(".." if Path.cwd().name == "notebooks" else ".")
IN_PARQUET = BASE_DIR / "data/processed/complaints.parquet"
OUT_DIR = BASE_DIR / "data/processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Working directory: {Path.cwd()}")
print(f"📁 Base directory: {BASE_DIR}")
print(f"📁 Input file: {IN_PARQUET}")
print(f"📁 Output directory: {OUT_DIR}")

# Cargar datos
df = pd.read_parquet(IN_PARQUET)
print(f"✅ Total registros cargados: {len(df):,}")
print(f"\n📊 Distribución PROD_TYPE:")
print(df['PROD_TYPE'].value_counts())

# Verificar YEARTXT = 9999
if 'YEARTXT' in df.columns:
    invalid_years = df[df['YEARTXT'] == 9999]
    print(f"\n⚠️  YEARTXT = 9999 (invalid): {len(invalid_years):,} registros")
    if len(invalid_years) > 0:
        print("  Estos registros se excluirán del filtrado final")


📁 Working directory: c:\Users\moral\tec_final\notebooks
📁 Base directory: ..
📁 Input file: ..\data\processed\complaints.parquet
📁 Output directory: ..\data\processed
✅ Total registros cargados: 2,137,711

📊 Distribución PROD_TYPE:
PROD_TYPE
V    2066945
T      40758
C      14995
E      14973
N          2
Name: count, dtype: int64

⚠️  YEARTXT = 9999 (invalid): 0 registros


In [17]:
# Análisis de marcas por tipo de producto
c_e_makes = set(df[df['PROD_TYPE'].isin(['C', 'E'])]['MAKETXT'].dropna().str.strip().str.upper())
v_makes = set(df[df['PROD_TYPE'] == 'V']['MAKETXT'].dropna().str.strip().str.upper())
t_makes = set(df[df['PROD_TYPE'] == 'T']['MAKETXT'].dropna().str.strip().str.upper())

print(f"Marcas en C/E: {len(c_e_makes)}")
print(f"Marcas en V: {len(v_makes)}")
print(f"Marcas en T: {len(t_makes)}")

# Marcas exclusivas de C/E (no aparecen en V ni T)
exclusive_c_e = c_e_makes - v_makes - t_makes
print(f"\nMarcas exclusivas de C/E (para exclusión): {len(exclusive_c_e)}")
print("Ejemplos:", sorted(list(exclusive_c_e))[:30])


Marcas en C/E: 875
Marcas en V: 1046
Marcas en T: 350

Marcas exclusivas de C/E (para exclusión): 727
Ejemplos: ['20TH CENTURY', '3M', '4WHEELS PARTS', 'A-1 ALTERNATIVE FUEL SYST', 'A.L. SOLUTIONS', 'A2ZEV', 'AAC', 'AAI MOTORSPORTS', 'ABADDON PRODUCTS', 'ABS-ACT-35 3-PT LWM 12', 'AC DELCO', 'ACC', 'ACCESSORY', 'ACCESSORY DISTRIBUTORS', 'ACCU-FAB', 'ACCU-FLOW', 'ACCURIDE', 'ACD TRIDON', 'ACE', 'ACE ELECTRIC', 'ACH', 'ACTIBRAKE', 'ACTIVHEAT', 'ACTRON', 'ADAPTIVE EQUIPMENT', 'ADCO PRODUCTS, INC.', 'ADV BRK SYS', 'ADVANTAGE', 'AEM', 'AEV']


In [18]:
# Guardar lista de marcas exclusivas para exportación
exclusive_makes_list = sorted(list(exclusive_c_e))
with open(OUT_DIR / "excluded_makes_from_c_e.json", "w") as f:
    json.dump(exclusive_makes_list, f, indent=2)

print(f"✅ Lista de marcas excluibles guardada: {len(exclusive_makes_list)} marcas")


✅ Lista de marcas excluibles guardada: 727 marcas


In [19]:
# PASO 1: Filtrar por PROD_TYPE (V y T) y YEARTXT válido
df_filtered = df[df['PROD_TYPE'].isin(['V', 'T'])].copy()

# Filtrar YEARTXT inválido (9999) si existe
if 'YEARTXT' in df_filtered.columns:
    df_filtered = df_filtered[df_filtered['YEARTXT'] != 9999].copy()

print(f"📊 Registros originales: {len(df):,}")
print(f"✅ Registros filtrados (V+T, YEARTXT≠9999): {len(df_filtered):,}")
print(f"❌ Registros excluidos (C+E+N+YEARTXT=9999): {len(df) - len(df_filtered):,}")

print(f"\n📈 Distribución en filtrado:")
print(df_filtered['PROD_TYPE'].value_counts())


📊 Registros originales: 2,137,711
✅ Registros filtrados (V+T, YEARTXT≠9999): 2,053,391
❌ Registros excluidos (C+E+N+YEARTXT=9999): 84,320

📈 Distribución en filtrado:
PROD_TYPE
V    2052035
T       1356
Name: count, dtype: int64


In [21]:
# PASO 2: DOWNSAMPLING POR COMPDESC (25% de cada componente)
# Estrategia: Para cada COMPDESC único, mantener solo 25% aleatorio

print(f"\n🔄 Aplicando downsampling (25% por COMPDESC)...")

# Obtener COMPDESC (componente principal, nivel L1)
if 'COMPDESC' in df_filtered.columns:
    df_filtered['COMP_L1'] = df_filtered['COMPDESC'].str.split(':').str[0]
else:
    # Si no existe COMPDESC, no hay downsampling
    df_filtered['COMP_L1'] = 'UNKNOWN'
    print("⚠️  No se encontró columna COMPDESC, omitiendo downsampling")
    df_filtered['COMP_L1'] = df_filtered.get('COMP_L1', 'UNKNOWN')

# Agrupar por COMP_L1 y muestrear 25% de cada grupo
print(f"📋 Componentes únicos (COMP_L1): {df_filtered['COMP_L1'].nunique()}")

df_downsampled = df_filtered.groupby('COMP_L1', group_keys=False).apply(
    lambda x: x.sample(frac=0.25, random_state=42)
).reset_index(drop=True)

print(f"✅ Registros después de downsampling: {len(df_downsampled):,}")
print(f"   Reducción: {len(df_filtered) - len(df_downsampled):,} registros ({100*(len(df_filtered) - len(df_downsampled))/len(df_filtered):.1f}%)")

# Resumen por componente
print(f"\n📊 Top 15 componentes por frecuencia (después de downsampling):")
print(df_downsampled['COMP_L1'].value_counts().head(50))



🔄 Aplicando downsampling (25% por COMPDESC)...
📋 Componentes únicos (COMP_L1): 54


C:\Users\moral\AppData\Local\Temp\ipykernel_23308\1816594874.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_downsampled = df_filtered.groupby('COMP_L1', group_keys=False).apply(


✅ Registros después de downsampling: 513,314
   Reducción: 1,540,077 registros (75.0%)

📊 Top 15 componentes por frecuencia (después de downsampling):
COMP_L1
ELECTRICAL SYSTEM                                              58080
POWER TRAIN                                                    51297
AIR BAGS                                                       38240
ENGINE                                                         38062
STEERING                                                       34610
ENGINE AND ENGINE COOLING                                      29590
SERVICE BRAKES, HYDRAULIC                                      26241
UNKNOWN OR OTHER                                               26236
STRUCTURE                                                      25003
SUSPENSION                                                     21713
VEHICLE SPEED CONTROL                                          20554
SERVICE BRAKES                                                 18378
EXTERIOR LIGH

In [32]:
# PASO 3: EXCLUSIÓN ADICIONAL DE COMPONENTES INDESEADOS
print("\n" + "="*70)
print("PASO 3: EXCLUSIÓN DE COMPONENTES INDESEADOS")
print("="*70)

# Componentes específicos a excluir (manual)
EXCLUDE_COMPONENTS = [
    'CHILD SEAT',
    'TRAILER HITCHES',
    'COMMUNICATION',
    'Chest Clip, Buckle, Harness',  # por si aparece como COMP_L1
]

# Componentes con menos de 18 registros también se excluyen
component_counts = df_downsampled['COMP_L1'].value_counts()
components_under_18 = component_counts[component_counts < 18].index.tolist()

# Combinar exclusiones
all_exclusions = set(EXCLUDE_COMPONENTS) | set(components_under_18)

print(f"Componentes a excluir:")
print(f"  - Específicos: {len(EXCLUDE_COMPONENTS)}")
print(f"  - Con <18 registros: {len(components_under_18)}")
print(f"  - Total único a excluir: {len(all_exclusions)}")

print(f"\nLista completa de exclusiones:")
for comp in sorted(all_exclusions):
    count = component_counts.get(comp, 0)
    print(f"  - {comp}: {count} registros")

# Filtrar: mantener solo componentes que NO están en la lista de exclusión
df_final = df_downsampled[~df_downsampled['COMP_L1'].isin(all_exclusions)].copy()

records_removed = len(df_downsampled) - len(df_final)
print(f"\nRegistros eliminados por exclusiones de componentes: {records_removed}")
print(f"Registros finales: {len(df_final):,}")
print(f"Componentes únicos finales: {df_final['COMP_L1'].nunique()}")



PASO 3: EXCLUSIÓN DE COMPONENTES INDESEADOS
Componentes a excluir:
  - Específicos: 4
  - Con <18 registros: 8
  - Total único a excluir: 12

Lista completa de exclusiones:
  - CHILD SEAT: 300 registros
  - COMMUNICATION: 23 registros
  - Carry Handle, Shell, Base: 16 registros
  - Chest Clip, Buckle, Harness: 30 registros
  - I suspect the car seat is counterfeit: 1 registros
  - Insert, Padding: 6 registros
  - NONE: 1 registros
  - OTHER: 1 registros
  - Other/Unknown: 2 registros
  - SERVICE BRAKES, HYDRAULIC; AUTOHOLD BRAKE SYSTEM/BRAKE HOLD: 1 registros
  - TRAILER HITCHES: 201 registros
  - Tether, Lower Anchor (on car seat or vehicle): 7 registros

Registros eliminados por exclusiones de componentes: 589
Registros finales: 512,725
Componentes únicos finales: 39


In [33]:
# Guardar versión FINAL (filtrado + downsampled + exclusiones de componentes)
df_final.to_parquet(OUT_DIR / "complaints_filtered_downsampled.parquet", index=False)

print("\n" + "="*70)
print("ARCHIVO FINAL GUARDADO")
print("="*70)
print(f"Complaints finales guardados: {len(df_final):,} registros")
print(f"Archivo: {OUT_DIR / 'complaints_filtered_downsampled.parquet'}")
print(f"  V: {len(df_filtered[df_filtered['PROD_TYPE']=='V']):,}")
print(f"  T: {len(df_filtered[df_filtered['PROD_TYPE']=='T']):,}")

print(f"\n📉 DOWNSAMPLED (25% por COMPDESC):")
print(f"  Total: {len(df_downsampled):,}")
print(f"  V: {len(df_downsampled[df_downsampled['PROD_TYPE']=='V']):,}")
print(f"  T: {len(df_downsampled[df_downsampled['PROD_TYPE']=='T']):,}")
print(f"  Componentes únicos: {df_downsampled['COMP_L1'].nunique()}")

print(f"\n🎯 REDUCCIÓN FINAL:")
print(f"  De {len(df):,} → {len(df_downsampled):,} registros")
print(f"  Reducción total: {100 * (1 - len(df_downsampled)/len(df)):.1f}%")

print(f"\n🏷️  MARCAS EXCLUSIVAS DE C/E PARA EXCLUIR: {len(exclusive_makes_list)}")



ARCHIVO FINAL GUARDADO
Complaints finales guardados: 512,725 registros
Archivo: ..\data\processed\complaints_filtered_downsampled.parquet
  V: 2,052,035
  T: 1,356

📉 DOWNSAMPLED (25% por COMPDESC):
  Total: 513,314
  V: 512,972
  T: 342
  Componentes únicos: 51

🎯 REDUCCIÓN FINAL:
  De 2,137,711 → 513,314 registros
  Reducción total: 76.0%

🏷️  MARCAS EXCLUSIVAS DE C/E PARA EXCLUIR: 727


In [34]:
# Análisis estadístico final
print("\n" + "="*70)
print("RESUMEN FINAL COMPLETO")
print("="*70)

print(f"\nORIGINAL (complaints.parquet):")
print(f"  Total: {len(df):,}")
print(f"  V (Vehicles): {len(df[df['PROD_TYPE']=='V']):,}")
print(f"  T (Tires): {len(df[df['PROD_TYPE']=='T']):,}")
print(f"  C (Child Seats): {len(df[df['PROD_TYPE']=='C']):,}")
print(f"  E (Equipment): {len(df[df['PROD_TYPE']=='E']):,}")

print(f"\nFILTRADO (V+T, YEARTXT!=9999):")
print(f"  Total: {len(df_filtered):,}")
print(f"  V: {len(df_filtered[df_filtered['PROD_TYPE']=='V']):,}")
print(f"  T: {len(df_filtered[df_filtered['PROD_TYPE']=='T']):,}")

print(f"\nDOWNSAMPLED (25% por COMPDESC):")
print(f"  Total: {len(df_downsampled):,}")
print(f"  V: {len(df_downsampled[df_downsampled['PROD_TYPE']=='V']):,}")
print(f"  T: {len(df_downsampled[df_downsampled['PROD_TYPE']=='T']):,}")
print(f"  Componentes: {df_downsampled['COMP_L1'].nunique()}")

print(f"\nFINAL (con exclusiones de componentes):")
print(f"  Total: {len(df_final):,}")
print(f"  V: {len(df_final[df_final['PROD_TYPE']=='V']):,}")
print(f"  T: {len(df_final[df_final['PROD_TYPE']=='T']):,}")
print(f"  Componentes únicos: {df_final['COMP_L1'].nunique()}")

print(f"\nREDUCCIÓN FINAL:")
print(f"  De {len(df):,} → {len(df_final):,} registros")
print(f"  Reducción: {100 * (1 - len(df_final)/len(df)):.1f}% ({len(df) - len(df_final):,} registros eliminados)")

print(f"\nTop 15 componentes finales:")
print(df_final['COMP_L1'].value_counts().head(15))

print(f"\nMARCAS EXCLUSIVAS DE C/E: {len(exclusive_makes_list)}")
print(f"COMPONENTES EXCLUIDOS: {len(all_exclusions)}")



RESUMEN FINAL COMPLETO

ORIGINAL (complaints.parquet):
  Total: 2,137,711
  V (Vehicles): 2,066,945
  T (Tires): 40,758
  C (Child Seats): 14,995
  E (Equipment): 14,973

FILTRADO (V+T, YEARTXT!=9999):
  Total: 2,053,391
  V: 2,052,035
  T: 1,356

DOWNSAMPLED (25% por COMPDESC):
  Total: 513,314
  V: 512,972
  T: 342
  Componentes: 51

FINAL (con exclusiones de componentes):
  Total: 512,725
  V: 512,383
  T: 342
  Componentes únicos: 39

REDUCCIÓN FINAL:
  De 2,137,711 → 512,725 registros
  Reducción: 76.0% (1,624,986 registros eliminados)

Top 15 componentes finales:
COMP_L1
ELECTRICAL SYSTEM            58080
POWER TRAIN                  51297
AIR BAGS                     38240
ENGINE                       38062
STEERING                     34610
ENGINE AND ENGINE COOLING    29590
SERVICE BRAKES, HYDRAULIC    26241
UNKNOWN OR OTHER             26236
STRUCTURE                    25003
SUSPENSION                   21713
VEHICLE SPEED CONTROL        20554
SERVICE BRAKES               1